# CDCN dengan peta kedalaman PRNet
Notebook ini menerapkan CDCN theta 0.7, supervisi PRNet depth untuk asli dan nol untuk spoof,
augmentasi crop/flip pasangan, serta MSE + contrastive depth loss.
Baca docs/CDCN_DEPTH_COLAB.md untuk penyesuaian dataset dan hardware.

Upload **CDCN-PRNet-Depth-Colab.zip** ke My Drive. Aktifkan GPU melalui Runtime.
Training utama dimulai dari bobot acak. Gunakan checkpoint hasil pipeline depth baru untuk fine-tuning.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, shutil, subprocess, sys, os

drive.mount("/content/drive")
ROOT = Path("/content/cdcn-depth")
ZIP = Path("/content/drive/MyDrive/CDCN-PRNet-Depth-Colab.zip")
assert ZIP.is_file(), "Upload ZIP ke My Drive terlebih dahulu."
shutil.unpack_archive(str(ZIP), ROOT)

# Normalize a previous private archive's dataset directory by its manifests.
data_parent = ROOT / "data"
private_root = data_parent / "pribadi"
if not private_root.exists() and data_parent.exists():
    candidates = [p for p in data_parent.iterdir()
                  if p.is_dir() and (p / "train_only.csv").is_file()]
    if len(candidates) == 1:
        candidates[0].rename(private_root)

os.chdir(ROOT)

def run(*args):
    with subprocess.Popen([str(arg) for arg in args], cwd=ROOT,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError("Perintah gagal. Periksa pesan di atas sebelum melanjutkan.")

## Pasang dependensi
Training PyTorch memakai interpreter Colab. PRNet memakai Python 3.11 terpisah untuk
kompatibilitas dengan bobot TensorFlow lama. Ubah tipe runtime ke GPU sebelum training.

In [ ]:
%pip install -q tensorboard pyyaml opencv-python-headless matplotlib uv
import torch
assert torch.cuda.is_available(), "Aktifkan GPU untuk training utama."
print("GPU:", torch.cuda.get_device_name(0))
DEPTH_PYTHON = Path("/content/prnet-env/bin/python")
if not DEPTH_PYTHON.is_file():
    run(sys.executable, "-m", "uv", "venv", "--python", "3.11", "/content/prnet-env")
run(sys.executable, "-m", "uv", "pip", "install", "--python", DEPTH_PYTHON,
    "-r", "requirements-depth.txt")

## Uji kode sebelum persiapan data
Uji ini memeriksa arsitektur terhadap kode rujukan, augmentasi RGB/depth, gradien loss,
dan agregasi skor per video.

In [ ]:
run(sys.executable, "-m", "unittest", "discover", "-s", "tests",
    "-p", "test_depth_pipeline.py", "-v")

## Persiapkan peta kedalaman SiW
Tahap ini memakai PRNet terlatih dan bisa memerlukan waktu. Jika detector atau PRNet gagal
pada sebagian gambar, periksa daftar kegagalan yang dicetak. Setelah meninjau pengecualian,
ubah ACCEPT_REVIEWED_FAILURES menjadi True dan jalankan ulang sel.
Peta yang sudah berhasil dibuat akan dipakai kembali dari cache.

Untuk melanjutkan dari sesi sebelumnya, ubah RESTORE_DEPTH_BACKUP menjadi True jika backup sudah ada.

In [ ]:
RESTORE_DEPTH_BACKUP = False
ACCEPT_REVIEWED_FAILURES = False
backup_dir = Path("/content/drive/MyDrive/CDCN_Depth")
backup_dir.mkdir(parents=True, exist_ok=True)
if RESTORE_DEPTH_BACKUP:
    saved = backup_dir / "siw_prepared.zip"
    assert saved.is_file(), "Backup depth belum ditemukan."
    shutil.unpack_archive(str(saved), ROOT / "data/siw")
else:
    command = [DEPTH_PYTHON, "-u", "prepare_depth.py", "--root", "data/siw", "--namespace", "siw"]
    if ACCEPT_REVIEWED_FAILURES:
        command.append("--allow-failed")
    run(*command)
print((ROOT / "data/siw/depth_summary.json").read_text())

## Periksa contoh RGB, depth, dan overlay
Peta wajah asli harus memiliki variasi spasial yang mengikuti wajah.
Periksa juga depth_failures.json. Banyak frame dari satu video tetap satu kelompok rekaman.

In [ ]:
import csv
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

dataset_root = ROOT / "data/siw"
failure_file = dataset_root / "depth_failures.json"
failures = json.loads(failure_file.read_text())
print("Jumlah frame yang dikecualikan:", len(failures))
for row in failures[:10]:
    print(row)
with (dataset_root / "train_depth.csv").open() as stream:
    rows = [r for r in csv.DictReader(stream) if r["label"] == "1"][:4]
assert rows, "Belum tersedia peta asli."
fig, axes = plt.subplots(len(rows), 3, figsize=(10, 3 * len(rows)), squeeze=False)
for i, row in enumerate(rows):
    rgb = np.asarray(Image.open(dataset_root / row["path"]).convert("RGB"))
    depth = np.asarray(Image.open(dataset_root / row["depth_path"]))
    axes[i, 0].imshow(rgb)
    axes[i, 1].imshow(depth, cmap="gray", vmin=0, vmax=255)
    axes[i, 2].imshow(rgb)
    axes[i, 2].imshow(depth, cmap="magma", alpha=0.45, vmin=0, vmax=255)
    for ax in axes[i]:
        ax.axis("off")
axes[0, 0].set_title("RGB")
axes[0, 1].set_title("PRNet depth")
axes[0, 2].set_title("Overlay")
plt.tight_layout()
plt.show()

## Simpan depth ke Drive dan periksa split
Backup memuat gambar prepared, depth, CSV hasil, dan laporan. Backup ini membantu
menghindari pengulangan rekonstruksi setelah runtime terputus.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

def backup_prepared(name):
    data = ROOT / "data" / name
    target = backup_dir / f"{name}_prepared.zip"
    with ZipFile(target, "w", ZIP_DEFLATED) as archive:
        for path in data.rglob("*"):
            if path.is_file() and (
                "prepared" in path.relative_to(data).parts
                or path.name.endswith("_depth.csv")
                or path.name.startswith("depth_")
            ):
                archive.write(path, path.relative_to(data).as_posix())
    print("Backup tersimpan", target)

backup_prepared("siw")
run(sys.executable, "-u", "train.py", "--config", "config/CDCN_depth.yaml", "--check-data")

## Coba satu epoch
Uji setup ini memakai output smoke tersendiri. Setelah sel berhasil, jalankan training utama dari awal.

In [ ]:
import yaml
cfg = yaml.safe_load((ROOT / "config/CDCN_depth.yaml").read_text())
cfg["train"]["num_epochs"] = 1
cfg["output_dir"] = "/content/drive/MyDrive/CDCN_Depth/smoke"
cfg["log_dir"] = "experiments/depth/smoke"
(ROOT / "config/CDCN_depth_smoke.yaml").write_text(yaml.safe_dump(cfg))
run(sys.executable, "-u", "train.py", "--config", "config/CDCN_depth_smoke.yaml")

## Training SiW utama dari awal
Konfigurasi maksimum 1300 epoch, LR 1e-4, weight decay 5e-5, LR turun setengah
setiap 500 epoch. Batch fisik 8 dengan akumulasi 7 dipilih untuk satu GPU Colab.
Statistik BatchNorm mengikuti batch fisik 8. Kode menyimpan best.pth, latest.pth,
threshold validasi, dan history.jsonl setiap epoch.

Jika runtime terputus, jalankan setup dan pulihkan depth backup, lalu ubah RESUME menjadi True.

In [ ]:
RESUME = False
command = [sys.executable, "-u", "train.py", "--config", "config/CDCN_depth.yaml"]
if RESUME:
    command += ["--resume", "/content/drive/MyDrive/CDCN_Depth/siw/latest.pth"]
run(*command)

## Test SiW dengan threshold validasi yang tersimpan
Lakukan setelah checkpoint dan keputusan training ditetapkan. Output menggunakan unit video.

In [ ]:
run(sys.executable, "evaluate.py", "--config", "config/CDCN_depth.yaml",
    "--checkpoint", "/content/drive/MyDrive/CDCN_Depth/siw/best.pth",
    "--test-csv", "test_depth.csv")

## Persiapkan depth Dataset Pribadi
Sel ini menyediakan train_only dan train_mixed dengan validasi/test yang sama.
Periksa kegagalan serta split_report dari sumber. Pemisahan identitas Dataset Pribadi masih
memerlukan verifikasi rekaman sumber untuk pelaporan skripsi.

In [ ]:
ACCEPT_REVIEWED_FAILURES_PRIBADI = False
command = [DEPTH_PYTHON, "-u", "prepare_depth.py", "--root", "data/pribadi",
           "--namespace", "pribadi", "--csv", "train_only.csv", "train_mixed.csv",
           "val.csv", "test.csv"]
if ACCEPT_REVIEWED_FAILURES_PRIBADI:
    command.append("--allow-failed")
run(*command)
print((ROOT / "data/pribadi/depth_summary.json").read_text())
backup_prepared("pribadi")

## Fine-tuning dari checkpoint depth SiW yang baru
Profil adaptasi memakai LR 1e-5 dan maksimum 50 epoch sebagai pilihan awal eksperimen.
Masing-masing run mulai dari best.pth SiW depth yang sama.

In [ ]:
for variant in ("only", "mixed"):
    print("\nFine-tuning", variant, flush=True)
    run(sys.executable, "-u", "train.py",
        "--config", f"config/CDCN_depth_pribadi_{variant}.yaml",
        "--init-checkpoint", "/content/drive/MyDrive/CDCN_Depth/siw/best.pth")

## Test kedua hasil adaptasi
Setiap checkpoint memakai threshold yang dipilih dari validasinya.
Simpan JSON, riwayat training, konfigurasi, dan laporan cakupan depth untuk arsip skripsi.

In [ ]:
for variant in ("only", "mixed"):
    print("\nTest", variant, flush=True)
    run(sys.executable, "evaluate.py",
        "--config", f"config/CDCN_depth_pribadi_{variant}.yaml",
        "--checkpoint", f"/content/drive/MyDrive/CDCN_Depth/pribadi_{variant}/best.pth",
        "--test-csv", "test_depth.csv")

## Acuan dan lingkup implementasi
- Paper CDCN, khususnya bagian 3 dan 4.2. https://arxiv.org/html/2003.04092
- Kode rujukan. https://github.com/ZitongYu/CDCN/tree/master/CVPR2020_paper_codes
- PRNet. https://github.com/YadiraF/PRNet

Pipeline memakai depth PRNet, CDCN, paired geometry, dan dua komponen loss sesuai metode.
Custom split SiW, bbox YuNet, rendering mesh, batch akumulasi, serta prosedur seleksi
model lokal didokumentasikan di docs/CDCN_DEPTH_COLAB.md. Angka benchmark paper
memerlukan dataset dan protokol evaluasi yang sama. Skor model baru perlu diukur
setelah training selesai.